In [1]:
import json
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    default_data_collator,
    pipeline  
)  

In [2]:
train_path = "train-v1.1.json"
dev_path = "dev-v1.1.json" 

with open (train_path , 'r') as f:
    train_data = json.load(f)

with open(dev_path , 'r') as f:
    dev_data = json.load(f)
 
print('loaded')

loaded


# Convert Flat To Dataframe

In [3]:
def squad_to_dataframe(data):
    contexts = []
    questions = []
    answers = []

    for article in data["data"]:
        for paragraph in article["paragraphs"]:
            context = paragraph["context"]

            for qa in paragraph["qas"]:
                question = qa["question"]
                answer = qa["answers"][0]  # take first answer

                contexts.append(context)
                questions.append(question)
                answers.append(answer)

    return pd.DataFrame({
        "context": contexts,
        "question": questions,
        "answers": answers
    })

train_df = squad_to_dataframe(train_data)
dev_df = squad_to_dataframe(dev_data)

train_df.head() 

,context,question,answers
0,"Architecturally, the school has a Catholic cha...",To whom did the Virgin Mary allegedly appear i...,"{'answer_start': 515, 'text': 'Saint Bernadett..."
1,"Architecturally, the school has a Catholic cha...",What is in front of the Notre Dame Main Building?,"{'answer_start': 188, 'text': 'a copper statue..."
2,"Architecturally, the school has a Catholic cha...",The Basilica of the Sacred heart at Notre Dame...,"{'answer_start': 279, 'text': 'the Main Buildi..."
3,"Architecturally, the school has a Catholic cha...",What is the Grotto at Notre Dame?,"{'answer_start': 381, 'text': 'a Marian place ..."
4,"Architecturally, the school has a Catholic cha...",What sits on top of the Main Building at Notre...,"{'answer_start': 92, 'text': 'a golden statue ..."


In [4]:
train_dataset = Dataset.from_pandas(train_df)
dev_dataset = Dataset.from_pandas(dev_df)
train_dataset

Dataset({
    features: ['context', 'question', 'answers'],
    num_rows: 87599
})

# Use Distilbert Model

In [5]:
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

max_length = 384
doc_stride = 128 

# Span Extraction

In [6]:
def preprocess_function(examples):
    questions = [q.strip() for q in examples["question"]]

    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    offset_mapping = inputs.pop("offset_mapping")
    sample_mapping = inputs.pop("overflow_to_sample_mapping")

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = inputs["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)

        sequence_ids = inputs.sequence_ids(i)
        sample_index = sample_mapping[i]
        answer = examples["answers"][sample_index]

        start_char = answer["answer_start"]
        end_char = start_char + len(answer["text"])

        token_start_index = 0
        while sequence_ids[token_start_index] != 1:
            token_start_index += 1

        token_end_index = len(input_ids) - 1
        while sequence_ids[token_end_index] != 1:
            token_end_index -= 1

        while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
            token_start_index += 1
        start_positions.append(token_start_index - 1)

        while offsets[token_end_index][1] >= end_char:
            token_end_index -= 1
        end_positions.append(token_end_index + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions

    return inputs  

In [7]:
tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names,
)

tokenized_dev = dev_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dev_dataset.column_names,
)

Map:   0%|          | 0/87599 [00:00<?, ? examples/s]

Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

In [13]:
tokenized_train = tokenized_train.select(range(100))
tokenized_dev = tokenized_dev.select(range(50))

# Building the model

In [9]:
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForQuestionAnswering LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
qa_outputs.bias         | MISSING    | 
qa_outputs.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [14]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./qa_model",

    eval_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=2,   # 🔥 smaller batch
    per_device_eval_batch_size=2,

    num_train_epochs=2,

    weight_decay=0.01,

    logging_steps=100,

    save_strategy="no",   # 🔥 prevents checkpoints eating memory

    fp16=False,           # CPU safe
    dataloader_num_workers=0,  # IMPORTANT for Windows/Jupyter
)

In [15]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_dev,
    processing_class=tokenizer,   # ✅ NEW PARAM NAME
    data_collator=default_data_collator,
)

In [16]:
trainer.train()   

C:\Users\FHCS\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss
1,No log,4.737653
2,4.775205,4.441419


TrainOutput(global_step=100, training_loss=4.775204772949219, metrics={'train_runtime': 557.9807, 'train_samples_per_second': 0.358, 'train_steps_per_second': 0.179, 'total_flos': 19597965004800.0, 'train_loss': 4.775204772949219, 'epoch': 2.0})

In [30]:
qa_pipeline = pipeline(
    "question-answering",
    model=model,
    tokenizer=tokenizer
)

context = "My name is Abid Ali i am a software engineer in ntu" 
question = "Where Abid get his education?"

qa_pipeline(question=question, context=context)

{'score': 0.03756936173886061,
 'start': 11,
 'end': 51,
 'answer': 'Abid Ali i am a software engineer in ntu'}

In [23]:
# Save model weights + config
trainer.model.save_pretrained("./qa_model_final")

# Save tokenizer (important!)
tokenizer.save_pretrained("./qa_model_final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./qa_model_final\\tokenizer_config.json', './qa_model_final\\tokenizer.json')